# 🌊 Notebook 15: The Master Blend (Exp 18)
**Eksperimen 18 | Target: submission_15.csv**

## Konsep Inovasi: Triple-Threat Democratic Blend
Pada notebook 14, fitur *River Physics* kita sukses menembus rekor CV (1.63) dan menaikkan skor LB (1.618).
Namun, LightGBM sendirian sudah mencapai batas maksimalnya (*plateau*).
Karena sekarang data kita sudah sangat bersih dan kaya akan fitur beresolusi tinggi, kita membangkitkan kembali **XGBoost** dan **CatBoost**.

Tujuan: Melatih ketiga model ini secara terpisah, lalu menggabungkan tebakan akhirnya (*Direct Blend*) untuk meredam varians secara drastis saat memprediksi data masa depan.

## 0. Setup & Konfigurasi

In [ ]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import optuna
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 150)
pd.set_option('display.float_format', '{:.4f}'.format)
optuna.logging.set_verbosity(optuna.logging.WARNING)

sns.set_theme(style='whitegrid')

# Config
SEED          = 42
N_FOLDS       = 5
N_OPTUNA_LGBM = 50
N_OPTUNA_XGB  = 30
N_OPTUNA_CAT  = 30
EARLY_STOP    = 200
SCALE_KM      = 30.0   
OUTPUT_CSV    = '../submissions/submission_15.csv'
TRAIN_PATH    = '../data/raw/train.csv'
TEST_PATH     = '../data/raw/test.csv'
ENV_PATH      = '../data/raw/data_pendukung/data_lingkungan.csv'
COORDS_PATH   = '../data/raw/data_pendukung/koordinat_pos.csv'

## 1. Pemuatan Data

In [ ]:
train    = pd.read_csv(TRAIN_PATH)
test     = pd.read_csv(TEST_PATH)
env_data = pd.read_csv(ENV_PATH)
coords   = pd.read_csv(COORDS_PATH)

train['datetime']    = pd.to_datetime(train['datetime'])
test['datetime']     = pd.to_datetime(test['id'].str[:19])
test['nama_pos']     = test['id'].str[22:]
env_data['datetime'] = pd.to_datetime(env_data['datetime'])
overall_cutoff       = train['datetime'].max()

le_station = LabelEncoder()
le_station.fit(np.sort(env_data['nama_pos'].unique()))


## 2. Fitur Fisika Aliran Sungai (Global Rainfall Pivot)

In [ ]:
env_sort = env_data.sort_values(['nama_pos','datetime'])
macro_cols   = ['nino_34','mjo_phase','mjo_amplitude','mjo_active','rmm1','rmm2']
dynamic_cols = ['surface_pressure_hpa','pressure_msl_hpa','soil_moisture_0_7cm',
                'soil_moisture_7_28cm','soil_moisture_28_100cm','soil_moisture_100_255cm']
for c in macro_cols:
    env_sort[c] = env_sort.groupby('nama_pos')[c].ffill().bfill()
for c in dynamic_cols:
    env_sort[c] = env_sort.groupby('nama_pos')[c].apply(
        lambda x: x.interpolate(method='linear').bfill().ffill()
    ).reset_index(level=0, drop=True)

agg_funcs = {col:'mean' for col in env_sort.columns if col not in ['nama_pos','landcover_name','datetime']}
agg_funcs['rainfall_mm']='sum'; agg_funcs['rainfall_openmeteo_mm']='sum'; agg_funcs['rainfall_max_24h_mm']='max'
env_6h = env_sort.set_index('datetime').groupby(
    ['nama_pos', pd.Grouper(freq='6h', label='right', closed='right')]
).agg(agg_funcs).reset_index()

env_rain_only = env_6h[['datetime', 'nama_pos', 'rainfall_mm']].copy()
env_rain_only['st_code'] = le_station.transform(env_rain_only['nama_pos'])
pivot_rain = env_rain_only.pivot(index='datetime', columns='st_code', values='rainfall_mm').fillna(0)

roll_3d = pivot_rain.rolling('3D').sum()
roll_7d = pivot_rain.rolling('7D').sum()
roll_3d.columns = [f'global_rain_st{c}_3d' for c in roll_3d.columns]
roll_7d.columns = [f'global_rain_st{c}_7d' for c in roll_7d.columns]
global_rain_features = pd.concat([roll_3d, roll_7d], axis=1).reset_index()


## 3. Fitur Lokal & Spasial

In [ ]:
station_profile = train.groupby('nama_pos')['tma_mdpl'].agg(tma_mean='mean', tma_std='std').reset_index()
station_profile['tma_std'] = station_profile['tma_std'].fillna(1.0)

train_sorted = train.sort_values(['nama_pos', 'datetime'])
anchor_offsets = {'0h':'0H','24h':'24H','72h':'72H','168h':'168H','336h':'336H'}
anchor_records = []
for pos in train['nama_pos'].unique():
    pos_df = train_sorted[train_sorted['nama_pos'] == pos]
    row = {'nama_pos': pos}
    for label, offset in anchor_offsets.items():
        subset = pos_df[pos_df['datetime'] <= overall_cutoff - pd.Timedelta(offset)]
        row[f'tma_anchor_{label}'] = np.nan if subset.empty else subset.iloc[-1]['tma_mdpl']
    subset30 = pos_df[(pos_df['datetime'] >= overall_cutoff - pd.Timedelta('30D')) & (pos_df['datetime'] <= overall_cutoff)]['tma_mdpl']
    row['tma_trend_30d'] = np.polyfit(np.arange(len(subset30)), subset30.values, 1)[0] if len(subset30)>1 else 0
    anchor_records.append(row)
station_profile = pd.merge(station_profile, pd.DataFrame(anchor_records), on='nama_pos', how='left')

sp_coords = pd.merge(station_profile[['nama_pos']], coords, on='nama_pos', how='left')
lats, lons = sp_coords['latitude'].values, sp_coords['longitude'].values
def haversine_matrix(lats, lons):
    R = 6371.0
    lat_r = np.radians(lats[:,None] - lats[None,:])
    lon_r = np.radians(lons[:,None] - lons[None,:])
    a = (np.sin(lat_r/2)**2 + np.cos(np.radians(lats[:,None]))*np.cos(np.radians(lats[None,:]))*np.sin(lon_r/2)**2)
    return 2*R*np.arcsin(np.sqrt(np.clip(a,0,1)))
D = haversine_matrix(lats, lons)
np.fill_diagonal(D, np.inf)

anchor_vals = station_profile['tma_anchor_0h'].values.astype(float)
wmeans = []
for i in range(len(lats)):
    weights = np.exp(-D[i]/SCALE_KM)
    valid   = ~np.isnan(anchor_vals)
    w = weights * valid; ws = w.sum()
    wmeans.append(np.nansum(w*np.nan_to_num(anchor_vals))/ws if ws>0 else np.nan)
station_profile['spatial_tma_anchor_0h'] = wmeans

for w, label in [(4,'24h'),(12,'3d'),(28,'7d')]:
    env_6h[f'rainfall_roll_{label}'] = env_6h.groupby('nama_pos')['rainfall_mm'].transform(lambda x: x.rolling(w, min_periods=1).sum())
    env_6h[f'pressure_roll_{label}'] = env_6h.groupby('nama_pos')['surface_pressure_hpa'].transform(lambda x: x.rolling(w, min_periods=1).mean())

train['month'] = train['datetime'].dt.month
seasonal_profile = train.groupby(['nama_pos','month'])['tma_mdpl'].mean().reset_index().rename(columns={'tma_mdpl':'tma_seasonal_mean'})


## 4. Penggabungan Data

In [ ]:
test['tma_mdpl'] = np.nan
all_data = pd.concat([train.drop(columns=['month'], errors='ignore'), test], ignore_index=True)
all_data = all_data.sort_values(['nama_pos','datetime']).reset_index(drop=True)

all_data = pd.merge(all_data, env_6h, on=['datetime','nama_pos'], how='left')
all_data = pd.merge(all_data, coords[['nama_pos','latitude','longitude']], on='nama_pos', how='left')
all_data = pd.merge(all_data, station_profile, on='nama_pos', how='left')
all_data = pd.merge(all_data, global_rain_features, on='datetime', how='left')

all_data['month']      = all_data['datetime'].dt.month
all_data['day_of_year']= all_data['datetime'].dt.dayofyear
all_data['sin_month']  = np.sin(2*np.pi*all_data['month']/12)
all_data['cos_month']  = np.cos(2*np.pi*all_data['month']/12)
all_data['nama_pos_encoded'] = le_station.transform(all_data['nama_pos'])

all_data = pd.merge(all_data, seasonal_profile, on=['nama_pos','month'], how='left')

steps_ahead = np.clip((all_data['datetime'] - overall_cutoff) / pd.Timedelta('3H'), 0, None)
all_data['tma_anchor_0h_norm'] = (all_data['tma_anchor_0h'] - all_data['tma_mean']) / all_data['tma_std']
all_data['spatial_anchor_0h_norm'] = (all_data['spatial_tma_anchor_0h'] - all_data['tma_mean']) / all_data['tma_std']
all_data['seasonal_norm'] = (all_data['tma_seasonal_mean'] - all_data['tma_mean']) / all_data['tma_std']

all_data['decay_short']  = all_data['tma_anchor_0h_norm'] * np.exp(-steps_ahead/28)
all_data['decay_long']   = all_data['tma_anchor_0h_norm'] * np.exp(-steps_ahead/168)
all_data['spatial_decay']= all_data['spatial_anchor_0h_norm'] * np.exp(-steps_ahead/168)

train_mask = all_data['tma_mdpl'].notnull()
all_data['tma_normalized'] = np.nan
all_data.loc[train_mask,'tma_normalized'] = (all_data.loc[train_mask,'tma_mdpl'] - all_data.loc[train_mask,'tma_mean']) / all_data.loc[train_mask,'tma_std']

train_data = all_data[train_mask].sort_values('datetime').reset_index(drop=True)
test_data  = all_data[~train_mask].sort_values('datetime').reset_index(drop=True)

drop_cols = ['datetime','nama_pos','tma_mdpl','tma_normalized','id','landcover_name']
drop_cols += [f'tma_anchor_{l}' for l in anchor_offsets]
drop_cols += ['spatial_tma_anchor_0h', 'tma_mean', 'tma_std', 'tma_seasonal_mean']
features  = [c for c in train_data.columns if c not in drop_cols]
target    = 'tma_normalized'

X_full, y_full = train_data[features], train_data[target]
X_test = test_data[features]
tscv   = TimeSeriesSplit(n_splits=N_FOLDS)


## 5. Optuna Tuning (LGBM, XGBoost, CatBoost)

In [ ]:
print('=== TUNING LIGHTGBM ===')
def obj_lgb(trial):
    params = {
        'n_estimators':      1500,
        'learning_rate':     trial.suggest_float('learning_rate',     0.01, 0.08, log=True),
        'num_leaves':        trial.suggest_int('num_leaves',           31,    120),
        'max_depth':         trial.suggest_int('max_depth',             5,     10),
        'subsample':         trial.suggest_float('subsample',          0.6,   0.9),
        'colsample_bytree':  trial.suggest_float('colsample_bytree',   0.4,   0.8),
        'reg_alpha':         trial.suggest_float('reg_alpha',          0.1,   5.0, log=True),
        'subsample_freq': 1, 'random_state': SEED, 'verbose': -1
    }
    scores = []
    for tr, va in tscv.split(X_full):
        m = lgb.LGBMRegressor(**params)
        m.fit(X_full.iloc[tr], y_full.iloc[tr], eval_set=[(X_full.iloc[va], y_full.iloc[va])],
              callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
        scores.append(mean_squared_error(y_full.iloc[va], m.predict(X_full.iloc[va])))
    return np.mean(scores)
study_lgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_lgb.optimize(obj_lgb, n_trials=N_OPTUNA_LGBM)
best_lgb = study_lgb.best_params
best_lgb.update({'n_estimators':3000,'random_state':SEED,'verbose':-1,'subsample_freq':1})

print('\n=== TUNING XGBOOST ===')
def obj_xgb(trial):
    params = {
        'n_estimators':      1000,
        'learning_rate':     trial.suggest_float('learning_rate',     0.01, 0.08, log=True),
        'max_depth':         trial.suggest_int('max_depth',             4,     8),
        'subsample':         trial.suggest_float('subsample',          0.6,   0.9),
        'colsample_bytree':  trial.suggest_float('colsample_bytree',   0.4,   0.8),
        'reg_alpha':         trial.suggest_float('reg_alpha',          0.1,   5.0, log=True),
        'tree_method': 'hist', 'random_state': SEED
    }
    scores = []
    for tr, va in tscv.split(X_full):
        m = XGBRegressor(**params)
        m.fit(X_full.iloc[tr], y_full.iloc[tr], eval_set=[(X_full.iloc[va], y_full.iloc[va])],
              early_stopping_rounds=EARLY_STOP, verbose=False)
        scores.append(mean_squared_error(y_full.iloc[va], m.predict(X_full.iloc[va])))
    return np.mean(scores)
study_xgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_xgb.optimize(obj_xgb, n_trials=N_OPTUNA_XGB)
best_xgb = study_xgb.best_params
best_xgb.update({'n_estimators':2000,'random_state':SEED,'tree_method':'hist'})

print('\n=== TUNING CATBOOST ===')
def obj_cat(trial):
    params = {
        'iterations':        1000,
        'learning_rate':     trial.suggest_float('learning_rate',     0.01, 0.08, log=True),
        'depth':             trial.suggest_int('depth',                 4,     8),
        'l2_leaf_reg':       trial.suggest_float('l2_leaf_reg',        1.0,  10.0, log=True),
        'random_seed': SEED, 'verbose': False
    }
    scores = []
    for tr, va in tscv.split(X_full):
        m = CatBoostRegressor(**params)
        m.fit(X_full.iloc[tr], y_full.iloc[tr], eval_set=(X_full.iloc[va], y_full.iloc[va]),
              early_stopping_rounds=EARLY_STOP, verbose=False)
        scores.append(mean_squared_error(y_full.iloc[va], m.predict(X_full.iloc[va])))
    return np.mean(scores)
study_cat = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_cat.optimize(obj_cat, n_trials=N_OPTUNA_CAT)
best_cat = study_cat.best_params
best_cat.update({'iterations':2000,'random_seed':SEED,'verbose':False})


## 6. Training & Evaluasi Blending (OOF CV)

In [ ]:
preds_test_lgb = np.zeros(len(X_test))
preds_test_xgb = np.zeros(len(X_test))
preds_test_cat = np.zeros(len(X_test))

# Untuk melacak OOF
oof_lgb, oof_xgb, oof_cat, oof_true = [], [], [], []

print('Final K-Fold Training for The Master Blend...')
for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_full)):
    X_tr, X_va = X_full.iloc[tr_idx], X_full.iloc[va_idx]
    y_tr, y_va = y_full.iloc[tr_idx], y_full.iloc[va_idx]
    
    val_mean = train_data.iloc[va_idx]['tma_mean'].values
    val_std  = train_data.iloc[va_idx]['tma_std'].values
    
    # 1. LGBM
    m_lgb = lgb.LGBMRegressor(**best_lgb)
    m_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
    p_lgb = (m_lgb.predict(X_va) * val_std) + val_mean
    oof_lgb.extend(p_lgb)
    preds_test_lgb += m_lgb.predict(X_test) / N_FOLDS
    
    # 2. XGBoost
    m_xgb = XGBRegressor(**best_xgb)
    m_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], early_stopping_rounds=EARLY_STOP, verbose=False)
    p_xgb = (m_xgb.predict(X_va) * val_std) + val_mean
    oof_xgb.extend(p_xgb)
    preds_test_xgb += m_xgb.predict(X_test) / N_FOLDS
    
    # 3. CatBoost
    m_cat = CatBoostRegressor(**best_cat)
    m_cat.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=EARLY_STOP, verbose=False)
    p_cat = (m_cat.predict(X_va) * val_std) + val_mean
    oof_cat.extend(p_cat)
    preds_test_cat += m_cat.predict(X_test) / N_FOLDS
    
    # Target Absolute
    y_abs = (y_va.values * val_std) + val_mean
    oof_true.extend(y_abs)
    
    print(f'Fold {fold+1} | RMSE -> LGBM: {np.sqrt(mean_squared_error(y_abs, p_lgb)):.4f}, XGB: {np.sqrt(mean_squared_error(y_abs, p_xgb)):.4f}, CAT: {np.sqrt(mean_squared_error(y_abs, p_cat)):.4f}')

oof_lgb, oof_xgb, oof_cat, oof_true = np.array(oof_lgb), np.array(oof_xgb), np.array(oof_cat), np.array(oof_true)

print('\n=== HASIL CV KESELURUHAN ===')
print(f'LGBM Alone CV      : {np.sqrt(mean_squared_error(oof_true, oof_lgb)):.4f}')
print(f'XGBoost Alone CV   : {np.sqrt(mean_squared_error(oof_true, oof_xgb)):.4f}')
print(f'CatBoost Alone CV  : {np.sqrt(mean_squared_error(oof_true, oof_cat)):.4f}')

# The Blend (40% LGBM + 30% XGB + 30% CAT)
oof_blend = (0.4 * oof_lgb) + (0.3 * oof_xgb) + (0.3 * oof_cat)
blend_cv = np.sqrt(mean_squared_error(oof_true, oof_blend))
print(f'\n🚀 MASTER BLEND CV RMSE: {blend_cv:.4f}')

if blend_cv > 1.636:
    print('\n WARNING: Blend RMSE lebih buruk dari Exp 17 (1.636). Pertimbangkan untuk tidak submit atau ubah bobotnya!')
else:
    print('\n SUCCESS: Blend RMSE memecahkan rekor coy')


## 7. Pembuatan File Submisi

In [ ]:
test_mean = test_data['tma_mean'].values
test_std  = test_data['tma_std'].values

# Decode normalized predictions
final_lgb = (preds_test_lgb * test_std) + test_mean
final_xgb = (preds_test_xgb * test_std) + test_mean
final_cat = (preds_test_cat * test_std) + test_mean

# Apply Weights
final_tma = (0.4 * final_lgb) + (0.3 * final_xgb) + (0.3 * final_cat)

submission = pd.DataFrame({'id': test_data['id'], 'tma_mdpl': final_tma})
os.makedirs('../submissions', exist_ok=True)
submission.to_csv(OUTPUT_CSV, index=False)

print('Eksperimen 18 (Master Blend) selesai!')
print(f'Output tersimpan di: {OUTPUT_CSV}')
print(submission.head(10))